# Local Flip Compare Plot (+ Dense Baseline as Layer-Drop Proxy)

Load `outputs/*.tsv` and plot AWQ/WANDA local-flip comparisons with the dense baseline as a reference.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
ROOT = 'compression/outputs'
OUT_DIR = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)

master_path = os.path.join(ROOT, 'all_settings_master.tsv')
dense_path = os.path.join(ROOT, 'dense_baseline_track.tsv')

master = pd.read_csv(master_path, sep='\t')
dense = pd.read_csv(dense_path)

# Use only local flip rows at focus layer for fair compare.
m = master.copy()
if 'focus_layer' in m.columns:
    m = m[m['layer'] == m['focus_layer']]

# Keep selected settings only; drop wanda_2_4 by design.
keep_settings = ['awq_native', 'wanda_4_8', 'wanda_unstructured']
if 'setting' in m.columns:
    m = m[m['setting'].isin(keep_settings)]

m = m.copy()
m['setting'] = m['setting'].astype(str)
dense = dense.copy()
dense['setting'] = 'layer_drop_baseline'

# Ensure required metric columns exist.
required_cols = ['delta_over_x', 'delta_over_x_std', 'para_over_x', 'para_over_x_std', 'perp_over_x', 'perp_over_x_std']
for c in required_cols:
    if c not in m.columns:
        m[c] = float('nan')
    if c not in dense.columns:
        dense[c] = float('nan')

keep_cols = ['setting', 'component', 'layer', 'delta_over_x', 'delta_over_x_std', 'para_over_x', 'para_over_x_std', 'perp_over_x', 'perp_over_x_std']
df = pd.concat([m[keep_cols], dense[keep_cols]], ignore_index=True)
df = df.sort_values(['setting', 'layer', 'component']).drop_duplicates(['setting', 'layer', 'component'], keep='last')

print('Rows:', len(df))
print('Settings:', sorted(df['setting'].dropna().unique().tolist()))
df.head()

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

ROOT = 'compression/outputs'
OUT_DIR = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)

master_path = os.path.join(ROOT, 'all_settings_master.tsv')
master = pd.read_csv(master_path, sep='\t')

df = master.copy()
if 'focus_layer' in df.columns:
    df = df[df['layer'] == df['focus_layer']]

KEEP_SETTINGS = ['awq_native', 'wanda_2_4', 'wanda_4_8', 'wanda_unstructured']
df = df[df['setting'].isin(KEEP_SETTINGS)].copy()

for col in [
    'delta_over_x', 'delta_over_x_std',
    'para_over_x', 'para_over_x_std',
    'perp_over_x', 'perp_over_x_std',
]:
    if col not in df.columns:
        df[col] = float('nan')

keep_cols = [
    'setting', 'component', 'layer',
    'delta_over_x', 'delta_over_x_std',
    'para_over_x', 'para_over_x_std',
    'perp_over_x', 'perp_over_x_std',
]
df = (
    df[keep_cols]
    .sort_values(['setting', 'layer', 'component'])
    .drop_duplicates(['setting', 'layer', 'component'], keep='last')
)

print('Rows:', len(df))
print('Settings:', sorted(df['setting'].unique().tolist()))
df.head()


In [ ]:
METRICS = ['para_over_x', 'perp_over_x']
COMPONENTS = ['block_out', 'attn_out', 'mlp_out']
PLOT_ORDER = ['awq_native', 'wanda_unstructured', 'wanda_4_8', 'wanda_2_4']
DISPLAY_NAME = {
    'awq_native': 'Quantization',
    'wanda_2_4': '2:4',
    'wanda_4_8': '4:8',
    'wanda_unstructured': 'Unstructured',
}

Y_LABEL = {
    'delta_over_x': r'$\|\Delta\| / \|u_{\mathrm{base}}\|$',
    'para_over_x': r'$\|\Delta_{\parallel}\| / \|u_{\mathrm{base}}\|$',
    'perp_over_x': r'$\|\Delta_{\perp}\| / \|u_{\mathrm{base}}\|$',
}
EXCLUDE_EDGE_LAYERS = True

for comp in COMPONENTS:
    d = df[df['component'] == comp].copy()
    if d.empty:
        continue

    d = d[d['setting'].isin(PLOT_ORDER)]
    layers_all = sorted(d['layer'].unique().tolist())

    if EXCLUDE_EDGE_LAYERS and len(layers_all) >= 3:
        core_min, core_max = layers_all[1], layers_all[-2]
        d = d[(d['layer'] >= core_min) & (d['layer'] <= core_max)].copy()
        layers_all = sorted(d['layer'].unique().tolist())
    else:
        core_min = layers_all[0] if layers_all else None
        core_max = layers_all[-1] if layers_all else None

    for metric in METRICS:
        d_metric = d.dropna(subset=[metric]).copy()
        valid_settings = [s for s in PLOT_ORDER if not d_metric[d_metric['setting'] == s].empty]
        if len(valid_settings) < 2:
            print(f'Skip {comp} | {metric}: only {len(valid_settings)} curve(s).')
            continue

        fig, ax = plt.subplots(figsize=(12, 5.6))

        for setting in valid_settings:
            g = d_metric[d_metric['setting'] == setting].sort_values('layer')
            line = ax.plot(
                g['layer'],
                g[metric],
                '-',
                marker='o',
                linewidth=1.8,
                markersize=4,
                label=DISPLAY_NAME.get(setting, setting),
            )[0]

            std_col = f'{metric}_std'
            if std_col in g.columns and g[std_col].notna().any():
                y = g[metric].astype(float)
                ystd = g[std_col].fillna(0.0).astype(float)
                ax.fill_between(
                    g['layer'],
                    y - ystd,
                    y + ystd,
                    color=line.get_color(),
                    alpha=0.15,
                )

        if layers_all:
            ax.set_xlim(min(layers_all), max(layers_all))

        ax.margins(x=0.01, y=0.08)
        ax.set_xlabel('Layer', fontsize=14)
        ax.set_ylabel(Y_LABEL.get(metric, metric), fontsize=14)
        ax.grid(alpha=0.3)
        ax.tick_params(axis='both', labelsize=13)
        ax.legend(fontsize=12, ncol=4, frameon=True, framealpha=0.9, loc='upper right')
        fig.subplots_adjust(top=0.92)
        plt.show()
